In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path:/ /Users/tiffanyyan/Documents/GitHub/QuantBricks
Fixed Income Library is loaded.


### Create Build Method Collection

In [2]:
bm_list = []
# create yield curve build method
content_sofr = {
    'TARGET' : 'SOFR-1B',
    'OVERNIGHT INDEX SWAP' : 'USD-SOFR-OIS'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_INDEX', content_sofr))
# create yield curve funding build method
content_funding = {
    'TARGET' : 'SOFR-1B-FLAT',
    'SPREAD ZERO RATE' : 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_funding))
# create yield curve common build method 
content = {
    'TARGET' : 'USD',
    'FUNDING PARAMETERS' : 'USD-FUNDING-PARAMETERS',
    'SOLVER' : 'BRENTQ'}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_COMMON', content))
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)

### Create Data Collection

In [3]:
### ois swap
df_swap = pd.DataFrame([    
    ['2Y', 0.035],
    ['3Y', 0.037],
    ['4Y', 0.039]],
    columns=['axis1', 'values']).set_index('axis1')
data_swap = qfCreateData1D('OVERNIGHT INDEX SWAP', 'USD-SOFR-OIS', df_swap)
### spread zero rate
df_spread_zero_rate = pd.DataFrame([['1Y', 0.]], columns=['axis1', 'values']).set_index('axis1')
data_szr = qfCreateData1D('SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD', df_spread_zero_rate)
### dg
df_dg = pd.DataFrame([
    ['Overnight Index Swap', 'USD-SOFR-OIS', 'SOFR-1B-FLAT']],
    columns=['DATA TYPE', 'DATA CONVENTION', 'FUNDING IDENTIFIER'])
data_fpt = qfCreateDataGeneric('DATA GENERIC', 'USD-FUNDING-PARAMETERS', df_dg)
### pack up all data into data collection
data_collection = qfCreateDataCollection([data_szr, data_swap, data_fpt])

### Create Yield Curve and Val Param

In [4]:
value_date = '2026-02-11'
yc_model = qfCreateModel(value_date, 'YIELD_CURVE', data_collection, build_method_collection)
fi_vp = qfCreateValuationParameters('FUNDING INDEX PARAMETER', {'Funding Index' : 'SOFR-1B-FLAT'})
vp_collection = qfCreateValuationParametersCollection([fi_vp])

### Build Product RFR Swap (Calibration Instrument)

In [5]:
effective_date = '2026-02-11'
termination_date = '2029-02-12'
pay_offset = '2D'
on_index = 'SOFR-1B'
fixed_rate = 0.037
pay_or_rec = 'receive'
notional = 1e4
accrual_peroid = '1Y'
accrual_basis = 'ACT/360'
floating_leg_accrual_period = '1Y'
business_day_convention = 'F'
holiday_convention = 'USGS'
spread = 0
compounding_method = 'compound'
product_on_rfr_swap = qfCreateProductRFRSwap(
    effective_date,
    termination_date,
    pay_offset,
    on_index,
    fixed_rate,
    pay_or_rec,
    notional,
    accrual_peroid,
    accrual_basis)
qfDisplayProduct(product_on_rfr_swap)

,Name,Value
0,Product Type,PRODUCT_RFR_SWAP
1,Notional,10000.0
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-02-11
5,Termination Date,2029-02-12
6,Payment Offset,2D
7,ON Index,SOFRON Actual/360
8,Fixed Rate,0.037
9,Pay Or Receive,RECEIVE


### Test Valuation

In [6]:
### test pv and cash
report = qfCreateValueReport(yc_model, product_on_rfr_swap, vp_collection, 'pvdetailed')
pv_base = qfCreateValueReport(yc_model, product_on_rfr_swap, vp_collection, 'pv')[0][1]
display(report.display())

,Currency,Type,Value
0,USD,PV,-2.321485e-10
1,USD,CASH,0.000000e+00


In [7]:
### test par rate
par_rate = qfCreateValueReport(yc_model, product_on_rfr_swap, vp_collection, 'parrateorspread')
print(f'The swap par rate is: {par_rate:.2%}.')

The swap par rate is: 3.70%.


In [8]:
### test pv01
pv01 = qfCreateValueReport(yc_model, product_on_rfr_swap, vp_collection, 'pv01')
print(f'The pv01 of the swap is: {pv01:.2f}.')

The pv01 of the swap is: 28364.18.


In [9]:
### test cf report
cf_report = qfCreateValueReport(yc_model, product_on_rfr_swap, vp_collection, 'cashflowsreport')
cf_report.display().T

,0,1,2,3,4,5,6,7
PRODUCT_TYPE,PRODUCT_RFR_SWAP,PRODUCT_RFR_SWAP,PRODUCT_RFR_SWAP,PRODUCT_RFR_SWAP,PRODUCT_RFR_SWAP,PRODUCT_RFR_SWAP,PRODUCT_RFR_SWAP,PRODUCT_RFR_SWAP
VALUATION_ENGINE_TYPE,ValuationEngineProductRfrSwap,ValuationEngineProductRfrSwap,ValuationEngineProductRfrSwap,ValuationEngineProductRfrSwap,ValuationEngineProductRfrSwap,ValuationEngineProductRfrSwap,ValuationEngineProductRfrSwap,ValuationEngineProductRfrSwap
LEG_ID,0,0,0,0,1,1,1,1
CASHFLOW_ID,0,1,2,3,0,1,2,3
PAY_OR_RECEIVE,1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0,-1.0
NOTIONAL,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0
PAY_DATE,"February 17th, 2026","February 17th, 2027","February 16th, 2028","February 14th, 2029","February 17th, 2026","February 17th, 2027","February 15th, 2028","February 14th, 2029"
FORECASTED_AMOUNT,1.027778,375.138889,377.194444,374.111111,0.955559,354.915566,356.775638,417.234375
PV,1.027189,362.073334,351.611038,334.763212,0.955012,342.554361,332.614502,373.350899
DISCOUNG FACTOR,0.999427,0.965171,0.932174,0.894823,0.999427,0.965171,0.932279,0.894823


In [10]:
### test risk
risk = qfCreateValueReport(yc_model, product_on_rfr_swap, vp_collection, 'firstorderrisk')
df_risk = risk.display()
df_risk.VALUES = df_risk.VALUES.round(10)
display(df_risk)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,0.000000
1,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,2Y,,0.035,0.0001,0.000000
2,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,3Y,,0.037,0.0001,-2.836418
3,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,4Y,,0.039,0.0001,0.000000


In [11]:
# Step 1: build bumped up model
bump_size = 1e-4
df_swap_bumped = df_swap.copy()
df_swap_bumped.loc['3Y'] += bump_size
data_swap_bumped = qfCreateData1D('OVERNIGHT INDEX SWAP', 'USD-SOFR-OIS', df_swap_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr, data_swap_bumped, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_on_rfr_swap, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is -2.8361395777462803.


In [13]:
# Step 1: build bumped up model
bump_size = 1e-4
df_spread_zero_rate_bumped = df_spread_zero_rate.copy()
df_spread_zero_rate_bumped.loc['1Y'] += bump_size
data_szr_bumped = qfCreateData1D('SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD', df_spread_zero_rate_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr_bumped, data_swap, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_on_rfr_swap, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is 9.094947017729282e-13.


### Build Product RFR Swap (Not Calibration Instrument)

In [25]:
effective_date = '2026-10-17'
termination_date = '2030-04-17'
pay_offset = '2D'
on_index = 'SOFR-1B'
fixed_rate = 0.027
pay_or_rec = 'pay'
notional = 1e6
accrual_peroid = '1Y'
accrual_basis = 'ACT/360'
floating_leg_accrual_period = '1Y'
business_day_convention = 'F'
holiday_convention = 'USGS'
compounding_method = 'compound'
product_on_rfr_swap_no_calib = qfCreateProductRFRSwap(
    effective_date,
    termination_date,
    pay_offset,
    on_index,
    fixed_rate,
    pay_or_rec,
    notional,
    accrual_peroid,
    accrual_basis)
qfDisplayProduct(product_on_rfr_swap_no_calib)

,Name,Value
0,Product Type,PRODUCT_RFR_SWAP
1,Notional,1000000.0
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-10-17
5,Termination Date,2030-04-17
6,Payment Offset,2D
7,ON Index,SOFRON Actual/360
8,Fixed Rate,0.027
9,Pay Or Receive,PAY


In [26]:
### test pv
pv_base = qfCreateValueReport(yc_model, product_on_rfr_swap_no_calib, vp_collection, 'pv')[0][1]
print(f'Base pv is {pv_base:.2f}.')

Base pv is 41709.64.


In [27]:
### test risk
df_risk_no_calib = qfCreateValueReport(yc_model, product_on_rfr_swap_no_calib, vp_collection, 'firstorderrisk').display()
df_risk_no_calib.VALUES = df_risk_no_calib.VALUES.round(2)
display(df_risk_no_calib)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,-11.40
1,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,2Y,,0.035,0.0001,-70.35
2,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,3Y,,0.037,0.0001,-53.25
3,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,4Y,,0.039,0.0001,429.72


### Risk Validation

In [28]:
risk_tenor = '2Y'
# Step 1: build bumped up model
bump_size = 1e-4
df_swap_bumped = df_swap.copy()
df_swap_bumped.loc[risk_tenor] += bump_size
data_swap_bumped = qfCreateData1D('OVERNIGHT INDEX SWAP', 'USD-SOFR-OIS', df_swap_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr, data_swap_bumped, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_on_rfr_swap_no_calib, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.2f}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is -70.34.


In [29]:
risk_tenor = '3Y'
# Step 1: build bumped up model
bump_size = 1e-4
df_swap_bumped = df_swap.copy()
df_swap_bumped.loc[risk_tenor] += bump_size
data_swap_bumped = qfCreateData1D('OVERNIGHT INDEX SWAP', 'USD-SOFR-OIS', df_swap_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr, data_swap_bumped, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_on_rfr_swap_no_calib, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.2f}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is -53.26.


In [30]:
risk_tenor = '4Y'
# Step 1: build bumped up model
bump_size = 1e-4
df_swap_bumped = df_swap.copy()
df_swap_bumped.loc[risk_tenor] += bump_size
data_swap_bumped = qfCreateData1D('OVERNIGHT INDEX SWAP', 'USD-SOFR-OIS', df_swap_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr, data_swap_bumped, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_on_rfr_swap_no_calib, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.2f}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is 429.66.


In [31]:
# Step 1: build bumped up model
bump_size = 1e-4
df_spread_zero_rate_bumped = df_spread_zero_rate.copy()
df_spread_zero_rate_bumped.loc['1Y'] += bump_size
data_szr_bumped = qfCreateData1D('SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD', df_spread_zero_rate_bumped)
data_collection_bumped = qfCreateDataCollection([data_szr_bumped, data_swap, data_fpt])
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_bumped, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_on_rfr_swap_no_calib, vp_collection, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is -11.39827069331659.
